# California Housing Price Prediction
## Part 1 - Data Loading, EDA & Data Preprocessing

In [ ]:
# Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

## Load Dataset

In [ ]:
housing = pd.read_csv("housing.csv")
housing.head()

## Basic Information

In [ ]:
print("Shape:", housing.shape)
display(housing.head())
display(housing.tail())
display(housing.sample(5))
housing.info()
housing.describe()

## Missing & Duplicate Values

In [ ]:
print(housing.isnull().sum())
print("Duplicate Rows:", housing.duplicated().sum())

plt.figure(figsize=(8,4))
sns.heatmap(housing.isnull(),cbar=False,cmap="viridis")
plt.title("Missing Values")
plt.show()

## Handle Missing Values

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")
housing["total_bedrooms"] = imputer.fit_transform(housing[["total_bedrooms"]])

print(housing.isnull().sum())

## Features & Target

In [ ]:
target="median_house_value"
X=housing.drop(target,axis=1)
y=housing[target]

print(X.select_dtypes(include=np.number).columns)
print(X.select_dtypes(exclude=np.number).columns)

## One Hot Encoding

In [ ]:
housing=pd.get_dummies(housing,columns=["ocean_proximity"],drop_first=True)
housing.head()

## Train Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X=housing.drop("median_house_value",axis=1)
y=housing["median_house_value"]

X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42)

print(X_train.shape,X_test.shape)

## Feature Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

print("Training shape:",X_train.shape)
print("Testing shape:",X_test.shape)

# Part 1 Completed

Topics covered:
- Import Libraries
- Load Dataset
- EDA
- Missing Values
- One-Hot Encoding
- Train/Test Split
- Feature Scaling

Next: Linear Regression, Decision Tree, Random Forest, Gradient Boosting.

# California Housing Price Prediction
## Part 2 - Model Training & Evaluation

Run Part 1 first so that `X_train`, `X_test`, `y_train`, and `y_test` are already available.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd
import numpy as np

## Helper Function

In [ ]:
def evaluate_model(model,name):
    model.fit(X_train,y_train)
    pred=model.predict(X_test)

    mae=mean_absolute_error(y_test,pred)
    mse=mean_squared_error(y_test,pred)
    rmse=np.sqrt(mse)
    r2=r2_score(y_test,pred)

    print(f"\n{name}")
    print("-"*40)
    print("MAE :",mae)
    print("MSE :",mse)
    print("RMSE:",rmse)
    print("R2  :",r2)

    return {
        "Model":name,
        "MAE":mae,
        "MSE":mse,
        "RMSE":rmse,
        "R2 Score":r2
    },pred

## Linear Regression

In [ ]:
result_Linear_Regression, pred_Linear_Regression = evaluate_model(LinearRegression(), "Linear Regression")

## Decision Tree

In [ ]:
result_Decision_Tree, pred_Decision_Tree = evaluate_model(DecisionTreeRegressor(random_state=42), "Decision Tree")

## Random Forest

In [ ]:
result_Random_Forest, pred_Random_Forest = evaluate_model(RandomForestRegressor(n_estimators=100, random_state=42), "Random Forest")

## Gradient Boosting

In [ ]:
result_Gradient_Boosting, pred_Gradient_Boosting = evaluate_model(GradientBoostingRegressor(random_state=42), "Gradient Boosting")

## Compare All Models

In [ ]:
results=pd.DataFrame([
result_Linear_Regression,
result_Decision_Tree,
result_Random_Forest,
result_Gradient_Boosting
])

results.sort_values("R2 Score",ascending=False)

## Actual vs Predicted (Best Model Example: Random Forest)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,6))
plt.scatter(y_test,pred_Random_Forest,alpha=0.5)
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Random Forest: Actual vs Predicted")
plt.show()

# Viva Notes

**MAE:** Average absolute error.

**MSE:** Average squared error.

**RMSE:** Square root of MSE; same unit as target.

**R² Score:** Closer to 1 means better prediction.

Generally, Random Forest and Gradient Boosting outperform a simple Linear Regression on this dataset.

# California Housing Price Prediction
## Part 3 - Visualization, Prediction & Model Saving

Run **Part 1** and **Part 2** first so trained models and variables are available.

## 1. Correlation Heatmap

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12,8))
sns.heatmap(housing.corr(numeric_only=True),
            cmap="coolwarm",
            annot=False)
plt.title("Correlation Heatmap")
plt.show()

## 2. Feature Importance (Random Forest)

In [ ]:
import pandas as pd

feature_names = X.columns

importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": RandomForestRegressor(n_estimators=100,random_state=42)
                    .fit(X_train,y_train)
                    .feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

display(importance.head(10))

In [ ]:
plt.figure(figsize=(10,5))
plt.bar(importance["Feature"][:10],
        importance["Importance"][:10])
plt.xticks(rotation=90)
plt.title("Top 10 Important Features")
plt.tight_layout()
plt.show()

## 3. Residual Plot

In [ ]:
residual = y_test - pred_Random_Forest

plt.figure(figsize=(6,5))
plt.scatter(pred_Random_Forest,residual,alpha=0.5)
plt.axhline(y=0,linestyle="--")
plt.xlabel("Predicted")
plt.ylabel("Residual")
plt.title("Residual Plot")
plt.show()

## 4. Predict House Price

In [ ]:
sample = X.iloc[[0]]

predicted_price = RandomForestRegressor(
    n_estimators=100,
    random_state=42
).fit(X_train,y_train).predict(sample)

print("Predicted House Price :",predicted_price[0])

## 5. Save Trained Model

In [ ]:
import joblib

rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train,y_train)

joblib.dump(rf_model,"housing_price_model.pkl")

print("Model Saved Successfully")

# Final Conclusion

- Random Forest generally provides the best performance on this dataset.
- Linear Regression is simple and fast but may underfit.
- Decision Tree can overfit.
- Gradient Boosting and Random Forest usually achieve higher R² scores.

## Viva Questions
1. What is Feature Importance?
2. Why do we save a model?
3. What is a Residual?
4. What does R² Score indicate?
5. Why is Random Forest often better than a single Decision Tree?


# California Housing Price Prediction
## Part 4 - Interactive Prediction, Report & Viva Preparation


# Project Abstract

The aim of this project is to predict California house prices using Machine Learning.
The project covers:
- Data Cleaning
- Data Preprocessing
- Model Training
- Model Evaluation
- Prediction
- Model Saving

Dataset: California Housing Dataset


## 1. Manual House Price Prediction

In [ ]:
# Example User Input
sample_data = {
    'longitude': -122.23,
    'latitude': 37.88,
    'housing_median_age': 41,
    'total_rooms': 880,
    'total_bedrooms': 129,
    'population': 322,
    'households': 126,
    'median_income': 8.3252
}

print("Example Input:")
print(sample_data)

print("\nNote:")
print("To predict your own house price, replace these values with your own dataset values.")

## 2. Model Comparison Summary

In [ ]:
comparison = {
"Linear Regression":"Simple, Fast, Easy to Understand",
"Decision Tree":"Can Overfit",
"Random Forest":"High Accuracy, Less Overfitting",
"Gradient Boosting":"Excellent Accuracy but Slower"
}

for model,desc in comparison.items():
    print(f"{model} --> {desc}")

## 3. Advantages of Random Forest

In [ ]:
advantages=[
"High Accuracy",
"Handles Missing Values Well",
"Less Overfitting",
"Works on Large Dataset",
"Feature Importance Available"
]

for i,a in enumerate(advantages,1):
    print(i,a)

## 4. Limitations

In [ ]:
limitations=[
"Training Time is High",
"Memory Usage is High",
"Less Interpretable than Linear Regression"
]

for i,l in enumerate(limitations,1):
    print(i,l)

## 5. Future Scope


Future Improvements:

- Hyperparameter Tuning
- Cross Validation
- XGBoost
- LightGBM
- Deep Learning
- Web App using Flask/Streamlit
- Deployment on Cloud


## 6. Viva Questions


1. What is Machine Learning?
2. What is Regression?
3. Why did we use California Housing Dataset?
4. What are Missing Values?
5. Why Median instead of Mean?
6. What is One-Hot Encoding?
7. Why StandardScaler?
8. What is Data Leakage?
9. Difference between MAE and RMSE?
10. What is R² Score?
11. Difference between Decision Tree and Random Forest?
12. Why Random Forest performed better?
13. What is Feature Importance?
14. What is Residual Error?
15. What is Model Saving?
16. What is Joblib?
17. What is Overfitting?
18. What is Underfitting?
19. What is Bias and Variance?
20. How can we improve the model?


## 7. Project Conclusion


Conclusion:

This project successfully predicts California house prices using different Machine Learning algorithms.

Among all algorithms, Random Forest generally provides the best balance between accuracy and robustness.

The project demonstrates the complete Machine Learning workflow:
1. Data Collection
2. Data Cleaning
3. Data Preprocessing
4. Model Training
5. Model Evaluation
6. Prediction
7. Model Saving



# Congratulations 🎉

You have now completed an end-to-end Machine Learning project.

Skills Covered:
- Python
- Pandas
- NumPy
- Matplotlib
- Seaborn
- Scikit-learn
- Data Cleaning
- Feature Engineering
- Regression
- Model Evaluation
- Visualization
